In [108]:
from audioop import avg

import pandas as pd
import simpful as sf
from numpy.f2py import rules
from simpful import Triangular_MF, FuzzySet, Gaussian_MF
from imblearn.under_sampling import RandomUnderSampler

from OccupancyFuzzyClassifier import  OccupancyFuzzyClassifier

df = pd.read_csv("data/Occupancy_Estimation.csv")
target_name ="Room_Occupancy_Count"
df = df.drop(columns=["Date", "Time"])

def under_sample(target_df):
    X = target_df.drop(columns=[target_name])
    y = target_df[target_name]
    rus = RandomUnderSampler(
        sampling_strategy={0: 600},
        random_state=42
    )

    X_res, y_res = rus.fit_resample(X, y)
    return  pd.concat([X_res, y_res], axis=1)
df = under_sample(df)
data_groups = {}
class_names = df[target_name].value_counts().index.tolist()
class_names = sorted(class_names)
for c in class_names:
    data_groups[c] = df[df[target_name] == c]


feature_names = [col for col in df.columns if col != target_name]

C:\Users\Krisent\PycharmProjects\SystemyRozmyteProjekt\.venv\lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
C:\Users\Krisent\PycharmProjects\SystemyRozmyteProjekt\.venv\lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


In [105]:





fuzzy_sets = {}
FS = sf.FuzzySystem(operators=['AND_PRODUCT'])
for feature in feature_names:
    fuzzy_sets[feature] = {}
    for c_name in class_names:
        avg = data_groups[c_name][feature].mean()
        std = max(data_groups[c_name][feature].std(),1e-6)
        fuzzy_sets[feature][c_name] = FuzzySet(function=Gaussian_MF(mu=avg,sigma=std),term=f"{c_name}_{feature}")
    FS.add_linguistic_variable(feature, sf.LinguisticVariable(list(fuzzy_sets[feature].values())))


output_sets = []
for c_name in class_names:
    spread = 0.05
    singleton = sf.FuzzySet(
        points=[[c_name - spread, 0], [c_name, 1], [c_name + spread, 0]],
        term=f"class_{c_name}"
    )
    output_sets.append(singleton)

FS.add_linguistic_variable(
    "occupancy",
    sf.LinguisticVariable(output_sets, universe_of_discourse=[0, 3]))

rules = []
for c_name in class_names:
    conds = []
    for feature in feature_names:
        conds.append(f"({feature} IS {c_name}_{feature})")
    conds = " AND ".join(conds)
    rule = f"IF ({conds}) THEN (occupancy IS class_{c_name})"
    rules.append(rule)
FS.add_rules(rules)



  ____  __  _  _  ____  ____  _  _  __   
 / ___)(  )( \/ )(  _ \(  __)/ )( \(  ) v2.12.0 
 \___ \ )( / \/ \ ) __/ ) _) ) \/ (/ (_/\ 
 (____/(__)\_)(_/(__)  (__)  \____/\____/

 https://github.com/aresio/simpful



In [106]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Split data
X = df[feature_names]
y = df[target_name]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Predict function
def predict(X_data):
    preds = []
    for idx, row in X_data.iterrows():
        for feat in feature_names:
            FS.set_variable(feat, row[feat])
        result = FS.Mamdani_inference(['occupancy'])
        preds.append(round(result['occupancy']))
    return preds
# Evaluate
y_pred = predict(X_test)
print(classification_report(y_test, y_pred))

SyntaxError: invalid syntax (4176486294.py, line 15)

In [109]:
clf = OccupancyFuzzyClassifier(under_sample_size=600)

X = clf.df[clf.feature_names]
y = clf.df[clf.target_name]
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

y_pred = clf.predict(X_test)


from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

C:\Users\Krisent\PycharmProjects\SystemyRozmyteProjekt\.venv\lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
C:\Users\Krisent\PycharmProjects\SystemyRozmyteProjekt\.venv\lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


  ____  __  _  _  ____  ____  _  _  __   
 / ___)(  )( \/ )(  _ \(  __)/ )( \(  ) v2.12.0 
 \___ \ )( / \/ \ ) __/ ) _) ) \/ (/ (_/\ 
 (____/(__)\_)(_/(__)  (__)  \____/\____/

 https://github.com/aresio/simpful

              precision    recall  f1-score   support

           0       1.00      0.90      0.95       185
           1       0.96      0.70      0.81       132
           2       0.75      0.64      0.69       217
           3       0.70      0.99      0.82       217

    accuracy                           0.81       751
   macro avg       0.85      0.80      0.82       751
weighted avg       0.83      0.81      0.81       751

